# 06b. Test-set lift table

Notebook 06 scored the selected logistic regression on the held-out test campaigns and saved one prediction per household-campaign pair.

This notebook turns those **saved predictions** into the view a marketing team uses: if we rank households by predicted redemption probability and contact only the top share, how many redeemers do we reach?

Nothing is refitted and no model is compared here. It is a second way of reading the single test evaluation already reported.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

current_folder = Path.cwd()
project_root = current_folder.parent if current_folder.name == "notebooks" else current_folder
evaluation_folder = project_root / "reports" / "model_evaluation"

predictions = pd.read_csv(evaluation_folder / "final_test_predictions.csv")

overall_rate = predictions["redeemed"].mean()
total_redeemers = int(predictions["redeemed"].sum())

print(f"Test examples: {len(predictions):,}")
print(f"Test campaigns: {predictions['CAMPAIGN'].nunique()}")
print(f"Recorded redeemers: {total_redeemers:,}")
print(f"Overall redemption rate: {overall_rate:.2%}")

Test examples: 2,218
Test campaigns: 7
Recorded redeemers: 317
Overall redemption rate: 14.29%


In [2]:
ranked = predictions.sort_values(
    ["predicted_redemption_probability", "household_key", "CAMPAIGN"],
    ascending=[False, True, True]
).reset_index(drop=True)

rows = []
for share in [0.10, 0.20, 0.30, 0.40, 0.50, 1.00]:
    top_n = int(round(len(ranked) * share))
    top = ranked.iloc[:top_n]
    rows.append({
        "top_share_contacted": share,
        "examples_contacted": top_n,
        "redeemers_reached": int(top["redeemed"].sum()),
        "redemption_rate": top["redeemed"].mean(),
        "lift_vs_overall": top["redeemed"].mean() / overall_rate,
        "share_of_redeemers_reached": top["redeemed"].sum() / total_redeemers,
    })

lift_table = pd.DataFrame(rows)
lift_table.round(4)

,top_share_contacted,examples_contacted,redeemers_reached,redemption_rate,lift_vs_overall,share_of_redeemers_reached
0,0.1,222,75,0.3378,2.3638,0.2366
1,0.2,444,122,0.2748,1.9226,0.3849
2,0.3,665,163,0.2451,1.7150,0.5142
3,0.4,887,192,0.2165,1.5145,0.6057
4,0.5,1109,224,0.2020,1.4132,0.7066
5,1.0,2218,317,0.1429,1.0000,1.0000


**Reading the table**

- *Redemption rate* is the share of contacted household-campaign pairs with a recorded redemption.
- *Lift* compares that rate with the overall test rate. A lift of 2 means twice the average.
- *Share of redeemers reached* is how many of all recorded redeemers fall in the contacted group.

A model with no ranking ability would reach about 10% of redeemers in the top 10%, 20% in the top 20%, and so on.

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

share_contacted = np.arange(1, len(ranked) + 1) / len(ranked)
share_reached = ranked["redeemed"].cumsum().to_numpy() / total_redeemers

fig, ax = plt.subplots(figsize=(6.5, 5))
ax.plot(share_contacted, share_reached, label="Logistic regression ranking")
ax.plot([0, 1], [0, 1], linestyle="--", color="gray", label="No ranking ability")
ax.set_xlabel("Share of test examples contacted, highest scores first")
ax.set_ylabel("Share of recorded redeemers reached")
ax.set_title("Cumulative gains on the held-out test campaigns")
ax.legend(frameon=False)
fig.tight_layout()

chart_path = evaluation_folder / "test_cumulative_gains.png"
fig.savefig(chart_path, dpi=150)
plt.close(fig)
print(f"Saved chart: {chart_path.relative_to(project_root)}")

Saved chart: reports/model_evaluation/test_cumulative_gains.png


In [4]:
lift_table.to_csv(evaluation_folder / "test_lift_table.csv", index=False)
print("Saved: reports/model_evaluation/test_lift_table.csv")

Saved: reports/model_evaluation/test_lift_table.csv


## Does the ranking work inside each campaign?

The pooled table mixes campaigns with different redemption rates, and the model knows the campaign type. To check that it also ranks households **within** a campaign, the next table takes the top 20% of each campaign separately.

In [5]:
by_campaign_rows = []
for campaign_id, group in predictions.groupby("CAMPAIGN"):
    group = group.sort_values(
        ["predicted_redemption_probability", "household_key"], ascending=[False, True]
    )
    top_n = max(1, int(round(len(group) * 0.20)))
    campaign_rate = group["redeemed"].mean()
    top_rate = group["redeemed"].iloc[:top_n].mean()
    by_campaign_rows.append({
        "CAMPAIGN": campaign_id,
        "examples": len(group),
        "redeemers": int(group["redeemed"].sum()),
        "campaign_redemption_rate": campaign_rate,
        "top_20pct_redemption_rate": top_rate,
        "top_20pct_lift_within_campaign": top_rate / campaign_rate,
    })

lift_by_campaign = pd.DataFrame(by_campaign_rows)
lift_by_campaign.to_csv(evaluation_folder / "test_lift_by_campaign.csv", index=False)
lift_by_campaign.round(3)

,CAMPAIGN,examples,redeemers,campaign_redemption_rate,top_20pct_redemption_rate,top_20pct_lift_within_campaign
0,18,1133,214,0.189,0.366,1.936
1,19,130,15,0.115,0.154,1.333
2,20,244,20,0.082,0.143,1.743
3,21,65,4,0.062,0.077,1.250
4,22,276,17,0.062,0.127,2.066
5,23,183,23,0.126,0.189,1.505
6,25,187,24,0.128,0.081,0.632


Six of the seven test campaigns show a within-campaign lift above 1. Campaign 25 is below 1. The smaller campaigns have few redeemers (4 to 24 each), so their figures are noisy.

## Limits

- The pooled ranking mixes seven campaigns. Campaign 18 (TypeA) supplies about half of the test examples and has the highest redemption rate, so the pooled figures lean on that one campaign.
- This measures who **redeems a coupon**. It does not show who buys more because of the campaign.
- It is one test period. The figures carry sampling uncertainty that is not quantified here.